# Intro to Unit Testing

For this notebook, we recommend splitting the VS Code editor. You can drag this notebook or the related files to a second panel while you work.

This notebook lives in `02-intro-to-unit-testing/`:

<pre>
02-intro-to-unit-testing/
├── 02-intro-to-unit-testing.ipynb
├── src/
│   └── unit_test_examples/
│      ├── __init__.py
│      ├── division.py
│      ├── email_generation_solution.py
│      ├── email_generation.py
│      ├── palindrome_solution.py
│      └── palindrome.py
└── tests/
   ├── test_division_solution.py
   ├── test_division.py
   ├── test_email_generation_solution.py
   ├── test_email_generation.py
   └── test_is_palindrome.py
</pre>

Unit testing usually becomes clearer in three steps: first you inspect behavior manually, then you express expectations with `assert`, and finally you let `pytest` organize those assertions into repeatable tests with useful failure messages. This progression matters because every step removes more guesswork. Manual checking can help while you explore, assertions make expectations explicit, and pytest turns those expectations into something you can re-run every time the code changes.

A good **unit test** is small, specific, and fast to understand. When one fails, you should be able to look at the test name and the assertion and quickly see which behavior no longer matches the contract.

This notebook focuses on three core ideas:

* writing clear assertions
* reducing repetition with parametrization
* sharing setup with fixtures

**What to expect:** the notebook examples and reference checks should pass immediately. The incomplete implementations in `palindrome.py` and `email_generation.py` are left in place by design, so their validation commands will fail until those modules are completed.

> All validation commands in this notebook assume your terminal is currently inside `02-intro-to-unit-testing/`.


In [ ]:
# Automatically reload modules when they are edited to avoid restarting the kernel.
%load_ext autoreload
%autoreload 2


In [ ]:
from pathlib import Path
import sys

for candidate in (
    Path.cwd().resolve(),
    Path.cwd().resolve() / "02-intro-to-unit-testing",
):
    if (candidate / "src").exists():
        NOTEBOOK_ROOT = candidate
        break
else:
    NOTEBOOK_ROOT = Path.cwd().resolve()

if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))


A **unit test** checks one small piece of behavior. We will start with a tiny function from `src/unit_test_examples/division.py`.

The goal is not just to see that the function works once. The goal is to describe what the function should do in a way that stays readable and repeatable as the codebase grows.


In [ ]:
from src.unit_test_examples.division import divide


In [ ]:
divide(6, 2), divide(3, 2), divide(-2, 0)


You *can* inspect behavior with `print()` statements, and they are still useful for quick debugging while you explore a problem.

They are not a strong long-term testing strategy, though. You have to scan the output yourself, and it becomes noisy once you add more cases or edge cases. A proper test should tell you automatically whether the behavior matches the expectation, without forcing you to read a block of output line by line.


In [ ]:
print("6 / 2 ->", divide(6, 2))
print("3 / 2 ->", divide(3, 2))
print("-2 / 0 ->", divide(-2, 0))


**Assertions** are more direct: each line says what should be true.

That makes them a better fit for repeatable checks, because Python stops immediately when one assumption is wrong. Instead of scanning output and guessing whether the result looks right, you write down the contract in code and let Python enforce it for you.


In [ ]:
assert divide(6, 2) == 3
assert divide(3, 2) == 1.5
assert divide(-2, 0) == "Cannot divide by zero"


## Pytest

`pytest` turns those assertions into named tests. In a notebook, `ipytest` gives us a small bridge so we can run pytest-style cells.

The important thing to notice is that pytest keeps each check inside a test function and reports exactly which test failed. That makes debugging much easier, especially once you have more than one case or more than one test file.


In [ ]:
import pytest
import ipytest

ipytest.autoconfig()


In [ ]:
%%ipytest -qq


def test_divide_basic():
    assert divide(6, 2) == 3
    assert divide(3, 2) == 1.5
    assert divide(-2, 0) == "Cannot divide by zero"


That test is already clearer than a block of `print()` calls. But each assertion still has the same shape: input, input, expected output.

When only the values change, **parametrization** is usually the clearest next step because it keeps the test logic in one place. Instead of copying the same test body several times, you write the structure once and supply the changing values as data.


In [ ]:
%%ipytest -qq


@pytest.mark.parametrize(
    "x, y, expected_result",
    [
        (6, 2, 3),
        (3, 2, 1.5),
        (-2, 0, "Cannot divide by zero"),
        (10, -2, -5),
    ],
)
def test_divide_parametrized(x, y, expected_result):
    assert divide(x, y) == expected_result


## Fixtures

Fixtures solve a different problem. Use them when **multiple tests need the same setup data**.

Instead of copying the same list or object into several tests, you define it once and let pytest pass it into each test that needs it. This keeps setup code centralized and makes later changes easier, because you only have to update that shared setup in one place.


In [ ]:
def format_data_for_display(people):
    return [
        f"{item['given_name']} {item['family_name']}: {item['title']}"
        for item in people
    ]


def format_data_for_excel(people):
    csv_string = "given,family,title\n"
    csv_string += "".join(
        f"{item['given_name']},{item['family_name']},{item['title']}\n"
        for item in people
    )
    return csv_string


In [ ]:
%%ipytest -qq


@pytest.fixture
def example_people_data():
    return [
        {
            "given_name": "Alfonsa",
            "family_name": "Ruiz",
            "title": "Senior Software Engineer",
        },
        {
            "given_name": "Sayid",
            "family_name": "Khan",
            "title": "Project Manager",
        },
    ]


def test_format_data_for_display(example_people_data):
    assert format_data_for_display(example_people_data) == [
        "Alfonsa Ruiz: Senior Software Engineer",
        "Sayid Khan: Project Manager",
    ]


def test_format_data_for_excel(example_people_data):
    assert format_data_for_excel(example_people_data) == (
        "given,family,title\n"
        "Alfonsa,Ruiz,Senior Software Engineer\n"
        "Sayid,Khan,Project Manager\n"
    )


**Rule of thumb:** use fixtures for shared setup, and use parametrization for input variations.

Those two tools often appear together, but they solve different kinds of repetition. Parametrization changes the test data while keeping the same logic, and fixtures keep the setup stable while multiple tests reuse it.

Python also has `doctest` for tiny executable examples in docstrings, but this notebook stays focused on `pytest`, since it is the main tool used throughout the repository files here.


## Exercise 1: Refactor Divide Tests

This is a readability task, not a behavior-change task.

1. Open `tests/test_division.py`.
2. Replace the repeated assertions with one parameterized pytest test.
3. Run `../.venv/bin/python -m pytest -q tests/test_division.py`.
4. Expect the test to pass before and after the refactor. The goal is cleaner structure, not different output.


In [ ]:
# @TODO Exercise 1: Refactor divide tests with parametrization.
# Objective: Replace repeated divide assertions with one parameterized pytest test.
# Edit files:
# - tests/test_division.py
# Validate with:
# - ../.venv/bin/python -m pytest -q tests/test_division.py
# Solution:
# - tests/test_division_solution.py


<details>
  <summary>Solution</summary>

```python
import pytest

from src.unit_test_examples.division import divide


@pytest.mark.parametrize(
    "x, y, expected_result",
    [
        (3, 2, 1.5),
        (5, 5, 1),
        (6, 2, 3),
        (-2, 0, "Cannot divide by zero"),
        (10, -2, -5),
    ],
)
def test_divide_parametrized(x, y, expected_result):
    assert divide(x, y) == expected_result
```
</details>


## Exercise 2: Palindrome Regression Suite

Implement `is_palindrome()` in `src/unit_test_examples/palindrome.py` so it handles casing, spaces, and punctuation.

1. Open `src/unit_test_examples/palindrome.py`.
2. Normalize the input before comparing it with its reverse.
3. Run `../.venv/bin/python -m pytest -q tests/test_is_palindrome.py`.
4. Expect this command to fail until the implementation is complete, then pass once the behavior matches the tests.


In [ ]:
# @TODO Exercise 2: Implement palindrome normalization.
# Objective: Make is_palindrome handle casing, spaces, and punctuation.
# Edit files:
# - src/unit_test_examples/palindrome.py
# Validate with:
# - ../.venv/bin/python -m pytest -q tests/test_is_palindrome.py
# Solution:
# - src/unit_test_examples/palindrome_solution.py


<details>
  <summary>Solution</summary>

```python
import re


def is_palindrome(s):
    normalized = s.lower()
    normalized = re.sub(r"[^A-Za-z0-9]+", "", normalized)
    return normalized == normalized[::-1]
```
</details>


## Exercise 3: Generate Neuefische Emails

Implement `generate_neuefische_emails()` in `src/unit_test_examples/email_generation.py`.

Expected format: `<first_name>.<last_name>@neuefische.de`

1. Open `src/unit_test_examples/email_generation.py`.
2. Build one email address per employee dictionary.
3. Normalize each name part with `strip()` and `lower()` before joining them.
4. Run `../.venv/bin/python -m pytest -q tests/test_email_generation.py`.
5. Expect this command to fail until the stub implementation is replaced.


In [ ]:
# @TODO Exercise 3: Generate normalized neuefische email addresses.
# Objective: Build one normalized email address per employee dictionary.
# Edit files:
# - src/unit_test_examples/email_generation.py
# Validate with:
# - ../.venv/bin/python -m pytest -q tests/test_email_generation.py
# Solution:
# - src/unit_test_examples/email_generation_solution.py


<details>
  <summary>Solution</summary>

```python
def generate_neuefische_emails(employees):
    return [
        f"{employee['first_name'].strip().lower()}.{employee['last_name'].strip().lower()}@neuefische.de"
        for employee in employees
    ]
```
</details>


## Running Tests

From inside `02-intro-to-unit-testing/`:

##### Validation commands:

```zsh
../.venv/bin/python -m pytest -q tests/test_division.py
../.venv/bin/python -m pytest -q tests/test_is_palindrome.py
../.venv/bin/python -m pytest -q tests/test_email_generation.py
```

**What to expect:**

* `tests/test_division.py` should already pass, because that task is about refactoring the test structure.
* `tests/test_is_palindrome.py` and `tests/test_email_generation.py` should fail until the incomplete implementations are finished.

##### Reference checks:

```zsh
../.venv/bin/python -m pytest -q tests/test_division_solution.py tests/test_email_generation_solution.py
```

##### Run all module tests:

```zsh
../.venv/bin/python -m pytest -q tests
```

Because this notebook intentionally includes unfinished implementation targets, `pytest -q tests` will fail until those files are completed.

Use `-s` when you want to see printed output.


### Quiz

1. **What is the main benefit of a unit test compared with checking output manually with `print()`?**  
    [ ] It makes Python run faster  
    [ ] It gives explicit pass/fail checks for small behavior  
    [ ] It installs missing packages automatically  
    [ ] It removes the need for debugging  

    <details>
      <summary>Show Answer</summary>

      **Correct Answer:** It gives explicit pass/fail checks for small behavior  
      **Description:** Unit tests state what should happen and fail loudly when behavior changes.
    </details>

---

2. **What problem does `@pytest.mark.parametrize()` solve best?**  
    [ ] Reusing shared setup data across many unrelated tests  
    [ ] Running the same test shape with different inputs  
    [ ] Installing pytest in a notebook  
    [ ] Hiding failing assertions  

    <details>
      <summary>Show Answer</summary>

      **Correct Answer:** Running the same test shape with different inputs  
      **Description:** Parametrization removes repeated test code when only the inputs and expected outputs change.
    </details>

---

3. **When is a fixture a better choice than parametrization?**  
    [ ] When several tests need the same setup data  
    [ ] When you only have one assertion  
    [ ] When you want to skip writing tests  
    [ ] When your function returns a string  

    <details>
      <summary>Show Answer</summary>

      **Correct Answer:** When several tests need the same setup data  
      **Description:** Fixtures keep shared setup in one place so multiple tests can reuse it.
    </details>

---

4. **Which command validates the email module implementation from inside the module folder?**  
    [ ] `python -m pytest -q tests/test_email_generation_solution.py`  
    [ ] `../.venv/bin/python -m pytest -q tests/test_email_generation.py`  
    [ ] `../.venv/bin/python email_generation.py`  
    [ ] `pytest src/unit_test_examples/email_generation.py`  

    <details>
      <summary>Show Answer</summary>

      **Correct Answer:** `../.venv/bin/python -m pytest -q tests/test_email_generation.py`  
      **Description:** The validation test imports `src/unit_test_examples/email_generation.py` directly.
    </details>
